In [1]:

import numpy as np

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator

from qiskit.circuit import ParameterVector
from qiskit import QuantumCircuit

# ============================================================
# Distances
# ============================================================

dists = [0.635, 0.735, 0.835, 0.935, 1.035]


def make_PES(dists, mol_data):
    ideal_vqe_energies = []
    for d in dists:

        print("======================================")
        print(f"H2 distance = {d} Å")
        print("======================================")

        qubit_op = mol_data[d]["qubit_op"]
        num_particles = mol_data[d]["num_particles"]
        num_spatial_orbitals = mol_data[d]["num_spatial_orbitals"]
        num_spin_orbitals = mol_data[d]["num_spin_orbitals"]
        nuclear_repulsion_energy = mol_data[d]["nuclear_repulsion_energy"]

        mapper = JordanWignerMapper()

        # ====================================================
        # Hartree-Fock
        # ====================================================

        hf_state = HartreeFock(
            num_spatial_orbitals,
            num_particles,
            mapper,
        )

        # ====================================================
        # Ansatz
        # ====================================================

    ## WRITE YOUR CODE BELOW HERE ##
        
        ansatz = UCCSD(
            num_spatial_orbitals,
            num_particles,
            mapper,
            initial_state=hf_state
        )


        # ====================================================
        # IDEAL STATEVECTOR VQE
        # ====================================================

        sv_estimator = StatevectorEstimator()

        optimizer = SLSQP()

        vqe = VQE(
            sv_estimator,
            ansatz,
            optimizer,
        )

        # Start at Hartree-Fock
        vqe.initial_point = np.zeros(
            ansatz.num_parameters
        )

        vqe_result = vqe.compute_minimum_eigenvalue(qubit_op)

        ideal_vqe_energy = vqe_result.eigenvalue.real + nuclear_repulsion_energy

        print(
            f"Ideal VQE energy   : "
            f"{ideal_vqe_energy:.8f} Ha"
        )
        ideal_vqe_energies.append(ideal_vqe_energy)
    return ideal_vqe_energies
